In [ ]:
# Run only if needed
%pip install openpyxl 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
DATA_DIR = Path('. Group Project/Post mid-term')

FILES = {
    "21_train": DATA_DIR / "kidney_final_train_dataset.csv",
    "21_test": DATA_DIR / "kidney_final_test_dataset.csv",
    "75": DATA_DIR / "75_Feature_Dataset.xlsx",
    "200": DATA_DIR / "200_Feature_Dataset.xlsx",
}

for name, path in FILES.items():
    print(name, "Exists:", path.exists())

In [ ]:
for name in ["75", "200"]:
    xls = pd.ExcelFile(FILES[name])
    print(f"\n{name}-feature workbook:")
    print(xls.sheet_names)
    xls.close()

In [ ]:
data = {}

data["21_train"] = pd.read_csv(FILES["21_train"])
data["21_test"] = pd.read_csv(FILES["21_test"])

for name in ["75", "200"]:
    sheets = pd.read_excel(
        FILES[name],
        sheet_name=None
    )

    for sheet_name, df in sheets.items():
        key = f"{name}::{sheet_name}"
        data[key] = df

for name, df in data.items():
    print(f"{name}: {df.shape}")

In [ ]:
TARGET = "icu_transfer"

ID_COLUMNS = {
    "subject_id",
    "hadm_id",
    "stay_id",
}

SPLIT_COLUMNS = {
    "dataset_split",
    "split",
    "set",
}

def audit_dataset(name, df):
    columns = set(df.columns)

    ids = sorted(columns & ID_COLUMNS)
    splits = sorted(columns & SPLIT_COLUMNS)

    excluded = ID_COLUMNS | SPLIT_COLUMNS | {TARGET}

    feature_cols = [
        c for c in df.columns
        if c not in excluded
    ]

    result = {
        "dataset": name,
        "rows": len(df),
        "total_columns": df.shape[1],
        "predictor_columns": len(feature_cols),
        "has_target": TARGET in columns,
        "identifiers": ", ".join(ids) if ids else "None",
        "split_columns": ", ".join(splits) if splits else "None",
        "missing_pct": round(
            df[feature_cols].isna().mean().mean() * 100, 2
        ) if feature_cols else np.nan,
        "duplicate_rows": df.duplicated().sum(),
    }

    if TARGET in columns:
        y = pd.to_numeric(df[TARGET], errors="coerce")

        result["positive_cases"] = (y == 1).sum()
        result["negative_cases"] = (y == 0).sum()
        result["positive_rate"] = round(
            (y == 1).mean() * 100, 3
        )
        result["invalid_targets"] = y.isna().sum() + (
            ~y.isin([0, 1]) & y.notna()
        ).sum()

    return result


audit_results = pd.DataFrame([
    audit_dataset(name, df)
    for name, df in data.items()
])

display(audit_results)

In [ ]:
for name, df in data.items():

    print(f"\n{'='*60}")
    print(name)

    available_ids = [
        c for c in ["subject_id", "hadm_id"]
        if c in df.columns
    ]

    if not available_ids:
        print("No patient/admission identifiers available.")
        continue

    for col in available_ids:
        print(f"\n{col}")
        print("Unique values:", df[col].nunique())
        print("Missing values:", df[col].isna().sum())
        print("Duplicate values:", df[col].duplicated().sum())

    if "hadm_id" in df.columns:
        duplicate_admissions = df.duplicated(
            subset=["hadm_id"]
        ).sum()

        print(
            "\nDuplicate admission IDs:",
            duplicate_admissions
        )

In [ ]:
for name, df in data.items():

    print(f"\n{name}")

    found = False

    for col in SPLIT_COLUMNS:
        if col in df.columns:
            found = True

            print(f"\nSplit column: {col}")

            display(
                df[col].value_counts(
                    dropna=False
                ).to_frame("count")
            )

            if TARGET in df.columns:
                display(
                    pd.crosstab(
                        df[col],
                        df[TARGET],
                        normalize="index"
                    ).round(4)
                )

    if not found:
        print("No explicit split column found.")

In [ ]:
EXCLUDE = ID_COLUMNS | SPLIT_COLUMNS | {TARGET}

feature_sets = {}

feature_sets["21"] = set(
    data["21_train"].columns
) - EXCLUDE

for size in ["75", "200"]:

    matching_keys = [
        key for key in data
        if key.startswith(f"{size}::")
    ]

    feature_sets[size] = set()

    for key in matching_keys:
        feature_sets[size].update(
            set(data[key].columns) - EXCLUDE
        )

for name, features in feature_sets.items():
    print(name, "features:", len(features))

In [ ]:
from itertools import combinations

overlap_results = []

for a, b in combinations(feature_sets, 2):

    A = feature_sets[a]
    B = feature_sets[b]

    intersection = A & B
    union = A | B

    overlap_results.append({
        "dataset_A": a,
        "dataset_B": b,
        "shared_features": len(intersection),
        "unique_to_A": len(A - B),
        "unique_to_B": len(B - A),
        "jaccard_similarity": (
            len(intersection) / len(union)
            if union else np.nan
        )
    })

overlap_df = pd.DataFrame(overlap_results)

display(overlap_df)

In [ ]:
names = ["21", "75", "200"]

matrix = pd.DataFrame(
    index=names,
    columns=names,
    dtype=float
)

for a in names:
    for b in names:

        A = feature_sets[a]
        B = feature_sets[b]

        union = A | B

        matrix.loc[a, b] = (
            len(A & B) / len(union)
            if union else np.nan
        )

plt.figure(figsize=(7, 5))

sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1
)

plt.title("Feature Set Similarity (Jaccard Index)")
plt.tight_layout()
plt.show()

In [ ]:
id_results = []

for a, b in combinations(data.keys(), 2):

    df_a = data[a]
    df_b = data[b]

    for id_col in ["hadm_id", "subject_id"]:

        if (
            id_col not in df_a.columns
            or id_col not in df_b.columns
        ):
            continue

        ids_a = set(df_a[id_col].dropna())
        ids_b = set(df_b[id_col].dropna())

        common = ids_a & ids_b

        id_results.append({
            "dataset_A": a,
            "dataset_B": b,
            "identifier": id_col,
            "unique_A": len(ids_a),
            "unique_B": len(ids_b),
            "shared": len(common),
            "overlap_A_pct": round(
                100 * len(common) / len(ids_a), 2
            ) if ids_a else np.nan,
            "overlap_B_pct": round(
                100 * len(common) / len(ids_b), 2
            ) if ids_b else np.nan,
        })

id_overlap_df = pd.DataFrame(id_results)

display(id_overlap_df)

In [ ]:
label_checks = []

for a, b in combinations(data.keys(), 2):

    df_a = data[a]
    df_b = data[b]

    required = {"hadm_id", TARGET}

    if not (
        required.issubset(df_a.columns)
        and required.issubset(df_b.columns)
    ):
        continue

    # Only compare admissions with exactly one row
    # in each dataset.
    unique_a = df_a[
        ~df_a["hadm_id"].duplicated(keep=False)
    ][["hadm_id", TARGET]]

    unique_b = df_b[
        ~df_b["hadm_id"].duplicated(keep=False)
    ][["hadm_id", TARGET]]

    merged = unique_a.merge(
        unique_b,
        on="hadm_id",
        suffixes=("_a", "_b"),
        how="inner"
    )

    mismatches = (
        merged[f"{TARGET}_a"]
        != merged[f"{TARGET}_b"]
    ).sum()

    label_checks.append({
        "dataset_A": a,
        "dataset_B": b,
        "matched_admissions": len(merged),
        "label_mismatches": mismatches,
    })

display(pd.DataFrame(label_checks))

In [ ]:
missing_results = []

for name, df in data.items():

    features = [
        c for c in df.columns
        if c not in EXCLUDE
    ]

    missing = (
        df[features]
        .isna()
        .mean()
        .sort_values(ascending=False)
    )

    missing_results.append({
        "dataset": name,
        "features_over_20pct_missing": (
            missing > 0.20
        ).sum(),
        "features_over_50pct_missing": (
            missing > 0.50
        ).sum(),
        "features_over_80pct_missing": (
            missing > 0.80
        ).sum(),
        "mean_missing_pct": round(
            missing.mean() * 100, 2
        )
    })

missing_summary = pd.DataFrame(missing_results)

display(missing_summary)

In [ ]:
with pd.ExcelWriter(
    "XGBoost_Dataset_Audit.xlsx",
    engine="openpyxl"
) as writer:

    audit_results.to_excel(
        writer,
        sheet_name="Dataset Summary",
        index=False
    )

    overlap_df.to_excel(
        writer,
        sheet_name="Feature Overlap",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="Jaccard Matrix"
    )

    id_overlap_df.to_excel(
        writer,
        sheet_name="Patient Overlap",
        index=False
    )

    pd.DataFrame(label_checks).to_excel(
        writer,
        sheet_name="Label Consistency",
        index=False
    )

    missing_summary.to_excel(
        writer,
        sheet_name="Missingness",
        index=False
    )

    feature_inventory = pd.DataFrame({
        "feature": sorted(
            set.union(*feature_sets.values())
        )
    })

    for name, features in feature_sets.items():
        feature_inventory[f"in_{name}"] = (
            feature_inventory["feature"].isin(features)
        )

    feature_inventory.to_excel(
        writer,
        sheet_name="Feature Inventory",
        index=False
    )

print("Audit saved successfully.")

In [ ]:
EXCLUDE = {
    "subject_id", "hadm_id", "stay_id",
    "dataset_split", "split", "set",
    "icu_transfer"
}

feature_sets = {
    "21": set(data["21_train"].columns) - EXCLUDE,
    "75": set(
        data["75::Final_75_Features"].columns
    ) - EXCLUDE,
    "200": set(
        data["200::Training Data"].columns
    ) - EXCLUDE
}

for name, features in feature_sets.items():
    print(name, len(features))

for a, b in combinations(feature_sets, 2):
    A, B = feature_sets[a], feature_sets[b]
    print(
        f"{a} vs {b}:",
        f"shared={len(A & B)},",
        f"unique to {a}={len(A-B)},",
        f"unique to {b}={len(B-A)}"
    )

In [ ]:
df75 = data["75::Final_75_Features"]
df200_train = data["200::Training Data"]
df200_test = data["200::Testing Data"]

ids75_train = set(
    df75.loc[df75["dataset_split"] == "train", "hadm_id"]
)
ids75_test = set(
    df75.loc[df75["dataset_split"] == "test", "hadm_id"]
)

ids200_train = set(df200_train["hadm_id"])
ids200_test = set(df200_test["hadm_id"])

print("Training splits identical:",
      ids75_train == ids200_train)

print("Testing splits identical:",
      ids75_test == ids200_test)

print("Training patients disjoint from testing:",
      set(df200_train["subject_id"]).isdisjoint(
          set(df200_test["subject_id"])
      ))

In [ ]:
# Combine the 200-feature training and test data
df200 = pd.concat(
    [df200_train, df200_test],
    ignore_index=True
)

# Identify shared predictor columns
shared_features = sorted(
    feature_sets["75"] & feature_sets["200"]
)

print("Shared features:", len(shared_features))

# Align patients by hospital admission
aligned = df75[
    ["hadm_id"] + shared_features
].merge(
    df200[["hadm_id"] + shared_features],
    on="hadm_id",
    suffixes=("_75", "_200"),
    validate="one_to_one"
)

print("Matched admissions:", len(aligned))

comparison = []

for feature in shared_features:

    a = aligned[f"{feature}_75"]
    b = aligned[f"{feature}_200"]

    # Missing values match only when both are missing
    both_missing = a.isna() & b.isna()

    # Numeric comparison with floating-point tolerance
    a_num = pd.to_numeric(a, errors="coerce")
    b_num = pd.to_numeric(b, errors="coerce")

    if a.notna().equals(a_num.notna()) and \
       b.notna().equals(b_num.notna()):

        equal = np.isclose(
            a_num.to_numpy(dtype=float),
            b_num.to_numpy(dtype=float),
            rtol=1e-6,
            atol=1e-8,
            equal_nan=True
        )

    else:
        equal = (
            a.eq(b).fillna(False) | both_missing
        ).to_numpy()

    comparison.append({
        "feature": feature,
        "matching_pct": 100 * np.mean(equal),
        "mismatched_rows": int((~equal).sum()),
        "missing_75_pct": 100 * a.isna().mean(),
        "missing_200_pct": 100 * b.isna().mean()
    })

feature_consistency = pd.DataFrame(comparison)

display(
    feature_consistency.sort_values("matching_pct")
)

print(
    "Features matching 100%:",
    (feature_consistency["mismatched_rows"] == 0).sum()
)

In [ ]:
diagnostics = []

for feature in shared_features:

    a = pd.to_numeric(
        aligned[f"{feature}_75"],
        errors="coerce"
    )

    b = pd.to_numeric(
        aligned[f"{feature}_200"],
        errors="coerce"
    )

    both_missing = a.isna() & b.isna()
    only_75 = a.notna() & b.isna()
    only_200 = a.isna() & b.notna()
    both_present = a.notna() & b.notna()

    different_values = (
        both_present
        & ~pd.Series(
            np.isclose(
                a.to_numpy(),
                b.to_numpy(),
                rtol=1e-6,
                atol=1e-8,
                equal_nan=True
            ),
            index=a.index
        )
    )

    diagnostics.append({
        "feature": feature,
        "both_missing": int(both_missing.sum()),
        "only_75_present": int(only_75.sum()),
        "only_200_present": int(only_200.sum()),
        "different_values": int(different_values.sum()),
        "both_present": int(both_present.sum()),
        "different_when_present_pct": round(
            100 * different_values.sum()
            / max(both_present.sum(), 1),
            3
        )
    })

diagnostics_df = pd.DataFrame(diagnostics)

display(
    diagnostics_df.sort_values(
        "different_values",
        ascending=False
    ).head(25)
)

In [ ]:
magnitude_results = []

for feature in shared_features:

    a = pd.to_numeric(
        aligned[f"{feature}_75"],
        errors="coerce"
    )

    b = pd.to_numeric(
        aligned[f"{feature}_200"],
        errors="coerce"
    )

    valid = a.notna() & b.notna()

    a_valid = a[valid]
    b_valid = b[valid]

    abs_diff = (a_valid - b_valid).abs()

    scale = np.maximum(
        np.maximum(a_valid.abs(), b_valid.abs()),
        1e-8
    )

    relative_diff = abs_diff / scale

    mismatched = ~np.isclose(
        a_valid.to_numpy(dtype=float),
        b_valid.to_numpy(dtype=float),
        rtol=1e-6,
        atol=1e-8
    )

    changed_abs = abs_diff.iloc[np.flatnonzero(mismatched)]
    changed_rel = relative_diff.iloc[np.flatnonzero(mismatched)]

    magnitude_results.append({
        "feature": feature,
        "n_changed": int(mismatched.sum()),
        "median_abs_diff_changed": (
            changed_abs.median()
            if len(changed_abs) else 0
        ),
        "p95_abs_diff_changed": (
            changed_abs.quantile(0.95)
            if len(changed_abs) else 0
        ),
        "median_relative_diff_changed_pct": (
            changed_rel.median() * 100
            if len(changed_rel) else 0
        ),
        "p95_relative_diff_changed_pct": (
            changed_rel.quantile(0.95) * 100
            if len(changed_rel) else 0
        )
    })

magnitude_df = pd.DataFrame(magnitude_results)

display(
    magnitude_df.sort_values(
        "n_changed",
        ascending=False
    ).head(25)
)

In [ ]:
%pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_curve,
    precision_recall_curve,
)

RANDOM_STATE = 42
TARGET = "icu_transfer"

In [ ]:
# 75-feature dataset
df75_train = df75[
    df75["dataset_split"] == "train"
].copy()

df75_test = df75[
    df75["dataset_split"] == "test"
].copy()

# 200-feature dataset
df200_train = df200_train.copy()
df200_test = df200_test.copy()

# Check admission IDs
assert set(df75_train["hadm_id"]) == set(
    df200_train["hadm_id"]
), "Training admission IDs differ!"

assert set(df75_test["hadm_id"]) == set(
    df200_test["hadm_id"]
), "Testing admission IDs differ!"

# Check patient-level separation
train_patients = set(df75_train["subject_id"])
test_patients = set(df75_test["subject_id"])

assert train_patients.isdisjoint(test_patients), (
    "Patient leakage between train and test!"
)

# Check unique admissions
for name, df in {
    "75 train": df75_train,
    "75 test": df75_test,
    "200 train": df200_train,
    "200 test": df200_test,
}.items():
    assert df["hadm_id"].is_unique, (
        f"Duplicate admission IDs in {name}"
    )

print("All split and admission checks passed.")

In [ ]:
def align_by_admission(reference, other):

    aligned_other = (
        other.set_index("hadm_id")
        .loc[reference["hadm_id"]]
        .reset_index()
    )

    assert np.array_equal(
        reference["hadm_id"].to_numpy(),
        aligned_other["hadm_id"].to_numpy()
    )

    assert np.array_equal(
        reference[TARGET].to_numpy(),
        aligned_other[TARGET].to_numpy()
    ), "Outcome labels differ!"

    return aligned_other


df200_train = align_by_admission(
    df75_train, df200_train
)

df200_test = align_by_admission(
    df75_test, df200_test
)

print("Training rows:", len(df75_train))
print("Testing rows:", len(df75_test))
print("Outcome labels verified.")

In [ ]:
features_75 = set(feature_sets["75"])
features_200 = set(feature_sets["200"])

shared_features = sorted(
    features_75 & features_200
)

unique_75 = sorted(
    features_75 - features_200
)

unique_200 = sorted(
    features_200 - features_75
)

print("Shared:", len(shared_features))
print("Unique to 75:", len(unique_75))
print("Unique to 200:", len(unique_200))

In [ ]:
def construct_feature_sets(df75_part, df200_part):

    # A: 73 shared features from 75
    A = df75_part[shared_features].copy()

    # B: 73 shared features from 200
    B = df200_part[shared_features].copy()

    # C: original 75
    C = df75_part[
        shared_features + unique_75
    ].copy()

    # D: original 200
    D = df200_part[
        shared_features + unique_200
    ].copy()

    # E: 73 shared from 75 + 127 unique from 200
    E = pd.concat(
        [
            df75_part[shared_features],
            df200_part[unique_200]
        ],
        axis=1
    )

    return {
        "A_73_from75": A,
        "B_73_from200": B,
        "C_75_original": C,
        "D_200_original": D,
        "E_200_hybrid": E
    }


X_train_sets = construct_feature_sets(
    df75_train.reset_index(drop=True),
    df200_train.reset_index(drop=True)
)

X_test_sets = construct_feature_sets(
    df75_test.reset_index(drop=True),
    df200_test.reset_index(drop=True)
)

y_train = df75_train[TARGET].reset_index(drop=True)
y_test = df75_test[TARGET].reset_index(drop=True)

groups_train = (
    df75_train["subject_id"]
    .reset_index(drop=True)
)

for name in X_train_sets:
    print(
        name,
        X_train_sets[name].shape,
        X_test_sets[name].shape
    )

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        X_train_sets["A_73_from75"],
        y_train,
        groups=groups_train
    )
)

print("Training observations:", len(train_idx))
print("Validation observations:", len(val_idx))

print(
    "Training prevalence:",
    y_train.iloc[train_idx].mean()
)

print(
    "Validation prevalence:",
    y_train.iloc[val_idx].mean()
)

print(
    "Test prevalence:",
    y_test.mean()
)

assert set(groups_train.iloc[train_idx]).isdisjoint(
    set(groups_train.iloc[val_idx])
)

In [ ]:
def prepare_numeric(X):

    X = X.copy()

    for col in X.columns:
        X[col] = pd.to_numeric(
            X[col],
            errors="raise"
        )

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return X.astype("float32")


for name in X_train_sets:

    X_train_sets[name] = prepare_numeric(
        X_train_sets[name]
    )

    X_test_sets[name] = prepare_numeric(
        X_test_sets[name]
    )

print("All feature matrices prepared.")

In [ ]:
XGB_PARAMS = {
    "n_estimators": 1000,
    "max_depth": 4,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_lambda": 5.0,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "early_stopping_rounds": 50,
}

In [ ]:
models = {}
results = []
predictions = {}
training_histories = {}

def calculate_metrics(y_true, probabilities):

    predictions_binary = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "ROC_AUC": roc_auc_score(
            y_true, probabilities
        ),
        "AUPRC": average_precision_score(
            y_true, probabilities
        ),
        "Precision": precision_score(
            y_true,
            predictions_binary,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions_binary,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            predictions_binary,
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            predictions_binary
        )
    }


for name, X_full in X_train_sets.items():

    print(f"\nTraining: {name}")

    X_tr = X_full.iloc[train_idx]
    X_val = X_full.iloc[val_idx]
    X_te = X_test_sets[name]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBClassifier(**XGB_PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False
    )

    models[name] = model
    training_histories[name] = model.evals_result()

    train_prob = model.predict_proba(X_tr)[:, 1]
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_te)[:, 1]

    predictions[name] = test_prob

    for split_name, y_true, prob in [
        ("Train", y_tr, train_prob),
        ("Validation", y_val, val_prob),
        ("Test", y_test, test_prob)
    ]:

        metrics = calculate_metrics(y_true, prob)

        results.append({
            "Model": name,
            "Split": split_name,
            "Features": X_full.shape[1],
            "Best_Iteration": model.best_iteration,
            **metrics
        })

    print(
        "Best iteration:",
        model.best_iteration
    )

    print(
        "Test AUPRC:",
        round(
            average_precision_score(
                y_test, test_prob
            ),
            5
        )
    )

In [ ]:
results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        ["Split", "AUPRC"],
        ascending=[True, False]
    )
)

In [ ]:
test_results = (
    results_df[
        results_df["Split"] == "Test"
    ]
    .sort_values("AUPRC", ascending=False)
)

display(
    test_results[
        [
            "Model",
            "Features",
            "ROC_AUC",
            "AUPRC",
            "Precision",
            "Recall",
            "F1",
            "Best_Iteration"
        ]
    ]
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

for name, prob in predictions.items():

    fpr, tpr, _ = roc_curve(
        y_test, prob
    )

    precision, recall, _ = precision_recall_curve(
        y_test, prob
    )

    auc = roc_auc_score(y_test, prob)
    ap = average_precision_score(y_test, prob)

    axes[0].plot(
        fpr, tpr,
        label=f"{name} ({auc:.3f})"
    )

    axes[1].plot(
        recall, precision,
        label=f"{name} ({ap:.3f})"
    )

axes[0].plot(
    [0, 1], [0, 1],
    "k--",
    alpha=0.5
)

axes[0].set_title("ROC Curves")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")

axes[1].axhline(
    y_test.mean(),
    linestyle="--",
    color="black",
    alpha=0.5,
    label="Positive prevalence"
)

axes[1].set_title("Precision–Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
comparison = results_df.pivot(
    index="Model",
    columns="Split",
    values=["ROC_AUC", "AUPRC"]
)

comparison.columns = [
    f"{metric}_{split}"
    for metric, split in comparison.columns
]

comparison["AUPRC_Gap"] = (
    comparison["AUPRC_Train"]
    - comparison["AUPRC_Validation"]
)

comparison["ROC_AUC_Gap"] = (
    comparison["ROC_AUC_Train"]
    - comparison["ROC_AUC_Validation"]
)

display(
    comparison.sort_values(
        "AUPRC_Validation",
        ascending=False
    )
)

In [ ]:
from sklearn.metrics import average_precision_score

test_groups = (
    df75_test["subject_id"]
    .reset_index(drop=True)
    .to_numpy()
)

def paired_bootstrap_ap(
    y_true,
    prob_a,
    prob_b,
    groups,
    n_bootstrap=1000,
    random_state=42
):

    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    prob_a = np.asarray(prob_a)
    prob_b = np.asarray(prob_b)
    groups = np.asarray(groups)

    unique_groups, inverse = np.unique(
        groups,
        return_inverse=True
    )

    group_indices = [
        np.flatnonzero(inverse == i)
        for i in range(len(unique_groups))
    ]

    differences = []

    for _ in range(n_bootstrap):

        sampled_groups = rng.integers(
            0,
            len(unique_groups),
            size=len(unique_groups)
        )

        idx = np.concatenate([
            group_indices[i]
            for i in sampled_groups
        ])

        y_boot = y_true[idx]

        if len(np.unique(y_boot)) < 2:
            continue

        ap_a = average_precision_score(
            y_boot, prob_a[idx]
        )

        ap_b = average_precision_score(
            y_boot, prob_b[idx]
        )

        differences.append(ap_a - ap_b)

    differences = np.asarray(differences)

    return {
        "Observed_Difference": (
            average_precision_score(y_true, prob_a)
            - average_precision_score(y_true, prob_b)
        ),
        "CI_Lower": np.percentile(
            differences, 2.5
        ),
        "CI_Upper": np.percentile(
            differences, 97.5
        )
    }

In [ ]:
comparisons = [
    ("A_73_from75", "B_73_from200"),
    ("C_75_original", "A_73_from75"),
    ("E_200_hybrid", "A_73_from75"),
    ("C_75_original", "D_200_original"),
    ("E_200_hybrid", "D_200_original")
]

bootstrap_results = []

for a, b in comparisons:

    result = paired_bootstrap_ap(
        y_test,
        predictions[a],
        predictions[b],
        test_groups
    )

    bootstrap_results.append({
        "Model_A": a,
        "Model_B": b,
        **result
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

display(bootstrap_df)

In [ ]:
with pd.ExcelWriter(
    "XGBoost_Feature_Comparison.xlsx",
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="All Metrics",
        index=False
    )

    test_results.to_excel(
        writer,
        sheet_name="Test Results",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Generalization"
    )

    bootstrap_df.to_excel(
        writer,
        sheet_name="Bootstrap Comparison",
        index=False
    )

print("Results exported successfully.")